# Traffic Intelligence

In [2]:
# !pip uninstall -y ultralytics torch torchvision torchaudio

!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

!pip install ultralytics==8.3.40 \
opencv-python-headless==4.10.0.84 \
websockets==15.0.1 \
yt-dlp==2024.8.6

Looking in indexes: https://download.pytorch.org/whl/cu121



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import asyncio
import base64
import json
import time
from datetime import datetime

import cv2
import torch
import websockets
from ultralytics import YOLO

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cpu


In [2]:
FILE_ID = "11xROvGqHG_aod4LEsp6LeWxPH1p6dbGZ"
!pip install gdown
!gdown "https://drive.google.com/file/d/11xROvGqHG_aod4LEsp6LeWxPH1p6dbGZ/view?usp=sharing" -O traffic.mp4

VIDEO_PATH = "traffic.mp4"

print("Video downloaded successfully")


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Video downloaded successfully


Downloading...
From: https://drive.google.com/uc?id=11xROvGqHG_aod4LEsp6LeWxPH1p6dbGZ
To: d:\Atri AI\Real-Time-Traffic-Intelligence-Dashboard\colab\traffic.mp4

  0%|          | 0.00/14.8M [00:00<?, ?B/s]
  4%|▎         | 524k/14.8M [00:00<00:04, 3.20MB/s]
  7%|▋         | 1.05M/14.8M [00:00<00:03, 4.05MB/s]
 14%|█▍        | 2.10M/14.8M [00:00<00:02, 4.37MB/s]
 18%|█▊        | 2.62M/14.8M [00:00<00:02, 4.39MB/s]
 21%|██        | 3.15M/14.8M [00:00<00:02, 4.34MB/s]
 28%|██▊       | 4.19M/14.8M [00:00<00:02, 5.02MB/s]
 35%|███▌      | 5.24M/14.8M [00:01<00:01, 5.68MB/s]
 42%|████▏     | 6.29M/14.8M [00:01<00:01, 6.61MB/s]
 53%|█████▎    | 7.86M/14.8M [00:01<00:00, 8.66MB/s]
 60%|██████    | 8.91M/14.8M [00:01<00:00, 8.97MB/s]
 71%|███████   | 10.5M/14.8M [00:01<00:00, 10.0MB/s]
 81%|████████▏ | 12.1M/14.8M [00:01<00:00, 10.9MB/s]
 92%|█████████▏| 13.6M/14.8M [00:01<00:00, 12.0MB/s]
100%|██████████| 14.8M/14.8M [00:01<00:00, 8.14MB/s]


In [3]:
WS_INGEST_URL = "wss://pessimist-dragonfly-eating.ngrok-free.dev/ws/ingest"

In [13]:
import threading

CLASSES = {"car", "truck", "bus", "motorcycle", "bicycle", "person"}

TARGET_FPS = 10.0
SEND_FPS = 15.0
OUTPUT_SIZE = (384, 216)
INFER_SIZE = 224
JPEG_QUALITY = 40
INFER_EVERY = 6
DRAW_BOXES = True
HALF_PRECISION = device.startswith("cuda")

model = YOLO("yolov8n.pt")
model.to(device)
CLASS_IDS = [cls_id for cls_id, name in model.names.items() if name in CLASSES]

def build_counts():
    return {"car": 0, "truck": 0, "bus": 0, "motorcycle": 0, "bicycle": 0, "person": 0}

def capture_worker(state, lock, stop_event):
    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    peak_density = 0
    last_frame_time = time.perf_counter()
    last_event_at = {"truck": 0, "bus": 0, "spike": 0}
    last_counts = build_counts()
    last_total = 0
    last_peak = 0
    frame_index = 0
    event_id = 0

    source_fps = cap.get(cv2.CAP_PROP_FPS)
    target_fps = min(source_fps, TARGET_FPS) if source_fps and source_fps > 0 else TARGET_FPS
    frame_interval = 1.0 / target_fps
    next_frame_at = time.perf_counter()

    try:
        while cap.isOpened() and not stop_event.is_set():
            now = time.perf_counter()
            if now < next_frame_at:
                time.sleep(next_frame_at - now)
            now = time.perf_counter()
            if now - next_frame_at > frame_interval:
                skip = int((now - next_frame_at) / frame_interval)
                for _ in range(skip):
                    if not cap.grab():
                        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                        break
                next_frame_at += skip * frame_interval

            ret, frame = cap.read()
            if not ret:
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                continue

            frame = cv2.resize(frame, OUTPUT_SIZE, interpolation=cv2.INTER_AREA)
            frame_index += 1
            do_infer = frame_index % INFER_EVERY == 0
            events = []

            if do_infer:
                results = model.predict(
                    source=frame,
                    device=device,
                    conf=0.4,
                    verbose=False,
                    imgsz=INFER_SIZE,
                    half=HALF_PRECISION,
                    classes=CLASS_IDS,
                    max_det=50,
                )
                result = results[0]

                counts = build_counts()
                for cls_id in result.boxes.cls.tolist():
                    label = result.names[int(cls_id)]
                    if label in counts:
                        counts[label] += 1

                total = sum(counts.values())
                peak_density = max(peak_density, total)
                last_counts = counts
                last_total = total
                last_peak = peak_density
                annotated = result.plot() if DRAW_BOXES else frame
            else:
                counts = last_counts
                total = last_total
                peak_density = last_peak
                annotated = frame

            now = time.perf_counter()
            fps = 1.0 / max(now - last_frame_time, 1e-6)
            last_frame_time = now
            ts = datetime.utcnow().isoformat()

            if do_infer:
                if counts["truck"] > 0 and now - last_event_at["truck"] > 3:
                    events.append({"message": "Truck detected", "severity": "info", "ts": ts})
                    last_event_at["truck"] = now
                if counts["bus"] > 0 and now - last_event_at["bus"] > 3:
                    events.append({"message": "Bus entered frame", "severity": "info", "ts": ts})
                    last_event_at["bus"] = now
                if peak_density > 0 and total / peak_density >= 0.85 and now - last_event_at["spike"] > 5:
                    events.append({"message": "Traffic spike detected", "severity": "critical", "ts": ts})
                    last_event_at["spike"] = now

            cv2.putText(
                annotated,
                f"FPS {fps:.1f}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 255),
                2,
            )

            _, buffer = cv2.imencode(".jpg", annotated, [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
            frame_b64 = base64.b64encode(buffer).decode("utf-8")
            event_id += 1

            with lock:
                state["frame_b64"] = frame_b64
                state["counts"] = counts
                state["total"] = total
                state["peak"] = peak_density
                state["fps"] = fps
                state["events"] = events
                state["event_id"] = event_id
                state["ts"] = ts
    finally:
        cap.release()

async def stream_video():
    state = {
        "frame_b64": None,
        "counts": build_counts(),
        "total": 0,
        "peak": 0,
        "fps": 0.0,
        "events": [],
        "event_id": 0,
        "ts": datetime.utcnow().isoformat(),
    }
    lock = threading.Lock()
    stop_event = threading.Event()
    capture_task = asyncio.create_task(asyncio.to_thread(capture_worker, state, lock, stop_event))
    last_event_id = -1
    send_interval = 1.0 / SEND_FPS
    next_send_at = time.perf_counter()

    try:
        async with websockets.connect(WS_INGEST_URL, max_size=2**23) as ws:
            while True:
                now = time.perf_counter()
                if now < next_send_at:
                    await asyncio.sleep(next_send_at - now)
                next_send_at = max(next_send_at + send_interval, time.perf_counter() + send_interval)

                with lock:
                    frame_b64 = state["frame_b64"]
                    counts = state["counts"]
                    total = state["total"]
                    peak_density = state["peak"]
                    fps = state["fps"]
                    events = state["events"]
                    event_id = state["event_id"]
                    ts = state["ts"]

                if not frame_b64:
                    continue

                send_events = events if event_id != last_event_id else []
                last_event_id = event_id

                payload = {
                    "timestamp": ts,
                    "counts": counts,
                    "totalVehicles": total,
                    "peakDensity": peak_density,
                    "fps": fps,
                    "events": send_events,
                    "frame": frame_b64,
                }

                await ws.send(json.dumps(payload))
    finally:
        stop_event.set()
        await capture_task

await stream_video()

C:\Users\hp\AppData\Local\Temp\ipykernel_8896\824106442.py:144: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "ts": datetime.utcnow().isoformat(),
C:\Users\hp\AppData\Local\Temp\ipykernel_8896\824106442.py:96: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().isoformat()


ConnectionClosedError: received 1012 (service restart); then sent 1012 (service restart)